## RAG EVALUATION (LangSmith)

Two things are evaluated separately, because a wrong answer can come from either stage:

| Stage | Question | Evaluators |
|---|---|---|
| **Retrieval** | Did we fetch the right documents? | `context_relevance` (LLM judge) |
| **Generation** | Is the answer good, given what was retrieved? | `correctness` (vs reference), `groundedness` (vs retrieved context), `answer_relevance` (vs question) |


In [2]:
import sys
sys.path.append('..')
from src.rag_pipeline import load_documents, build_hybrid_retriever, build_llm


d:\RAG_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
documents = load_documents()    
hybrid_retriever = build_hybrid_retriever()
llm = build_llm()

Loading embedding model: BAAI/bge-small-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1849.01it/s]


Model loaded. Embedding dimension: 384
Vector store ready. Collection: json_qa_documents (2964 documents)
BM25 index built over 2964 documents


In [4]:
import json, random, re, os
import openai
from langsmith import Client, traceable, wrappers

os.environ["LANGSMITH_TRACING"] = "true"
assert os.getenv("LANGSMITH_API_KEY"), "LANGSMITH_API_KEY missing from .env"

ls_client = Client()

# Judge client: Groq via its OpenAI-compatible endpoint (wrapped so judge calls are traced too)
judge_client = wrappers.wrap_openai(
    openai.OpenAI(
        api_key=os.getenv("EVAL_GROQ_API_KEY"),
        base_url="https://api.groq.com/openai/v1",
    )
)
# Ideally use a DIFFERENT model from the generator (llm above) to avoid self-preference bias.
JUDGE_MODEL = "openai/gpt-oss-20b"

### 1. Build the evaluation dataset
The corpus is already Q&A pairs with a `doc_id`, so we can build a labelled set automatically:


In [5]:
DATASET_NAME = "RAG Eval - Mental Health QA"
N_EXAMPLES = 10
random.seed(42)

def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks some reasoning models emit."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

def split_qa(page_content: str):
    q, a = page_content.split("\nAnswer: ", 1)
    return q.removeprefix("Question: ").strip(), a.strip()

def paraphrase(question: str) -> str:
    r = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.7,
        messages=[
            {"role": "system", "content": "Rewrite the user's question the way a real person might phrase it, "
                                          "using different words but the same meaning. Output ONLY the rewritten question."},
            {"role": "user", "content": question},
        ],
    )
    return strip_think(r.choices[0].message.content)

def build_examples(docs):
    examples = []
    for d in docs:
        q, a = split_qa(d.page_content)
        examples.append({
            "inputs":  {"question": paraphrase(q)},
            "outputs": {"answer": a, "doc_id": d.metadata["doc_id"]},
            "metadata": {
                "topic_group": d.metadata.get("topic_group"),
                "safety_sensitive": "safety_sensitive" in (d.metadata.get("flags") or []),
                "original_question": q,
            },
        })
    return examples

if ls_client.has_dataset(dataset_name=DATASET_NAME):
    print("Dataset already exists, reusing it.")
else:
    dataset = ls_client.create_dataset(DATASET_NAME, description="Paraphrased questions with gold answer + gold doc_id")
    ls_client.create_examples(dataset_id=dataset.id, examples=build_examples(random.sample(documents, N_EXAMPLES)))
    print(f"Created dataset with {N_EXAMPLES} examples.")

Dataset already exists, reusing it.


### Retrieve both docs and answer

In [6]:
def make_target(retriever, top_k: int = 3):
    
    @traceable(run_type="retriever", name="retrieve")
    def retrieve(query: str):
        docs = retriever.retrieve(query, top_k=top_k)
        return [
            {"page_content": d["content"], "type": "Document",
             "metadata": {"id": d["id"], "rank": d["rank"]}}
            for d in docs
        ]

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        docs = retrieve(question)
        context = "\n\n".join(d["page_content"] for d in docs)
        if not context:
            return {"response": "No relevant context found to answer the question.",
                    "retrieved_ids": [], "context": ""}

        prompt = (
            "Use the following context to answer the question concisely. "
            "Answer only from the context.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
        )
        response = llm.invoke(prompt).content
        return {
            "response": strip_think(response),
            "retrieved_ids": [d["metadata"]["id"] for d in docs],
            "context": context,
        }
    return target

### Evaluators or Metrics

1. **Correctness:** Response vs reference answer (`Does Answer Match the Groundtruth Answer?`)
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [7]:
from typing_extensions import Annotated, TypedDict

## Correctness Output Schema

## Grade output schema
class CorrectnessGrade(TypedDict):
    explanation: Annotated[str,...,'Explain your reasoning for the score']
    correct: Annotated[bool,...,'True if the answer is correct, False otherwise.']

## correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

from langchain_groq import ChatGroq

grader_llm=ChatGroq(model=JUDGE_MODEL,temperature=0,groq_api_key=os.getenv('EVAL_GROQ_API_KEY'),).with_structured_output(CorrectnessGrade)

## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['response']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return {'key':'correctness', 'score':int(grade['correct']), 'comment':grade['explanation']}

**2. Answer Relevance:** Response vs input (Does answer address the question?)
- We simply look at the inputs and outputs without needing the reference_outputs.

In [8]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

relevance_llm=ChatGroq(model=JUDGE_MODEL,temperature=0,groq_api_key=os.getenv('EVAL_GROQ_API_KEY'),).with_structured_output(RelevanceGrade)

def relevance(inputs: dict, outputs: dict) -> dict:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['response']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return {'key': 'relevance', 'score':int(grade['relevant']), 'comment':grade['explanation']}


**3. Groundness:** Response vs retrieved docs (Is the answer grounded in the documents?)

In [9]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm=ChatGroq(model=JUDGE_MODEL,temperature=0,groq_api_key=os.getenv('EVAL_GROQ_API_KEY'),).with_structured_output(GroundedGrade)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> dict:
    """A simple evaluator for RAG answer groundedness."""
    answer = f"FACTS: {outputs['context']}\nSTUDENT ANSWER: {outputs['response']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return {"key": "groundedness", "score": int(grade["grounded"]), "comment": grade["explanation"]}


**4. Retrieval Relevance:** Retrieved docs vs input (Are documents relevant to the questions?)

In [10]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
retriever_llm=ChatGroq(model=JUDGE_MODEL,temperature=0,groq_api_key=os.getenv('EVAL_GROQ_API_KEY'),).with_structured_output(RetrievalRelevanceGrade)

#Evaluator
def retriever_relevance(inputs: dict, outputs: dict) -> dict:
    """Evaluator for retrieval relevance: are the retrieved docs related to the question?"""
    text = f"QUESTION: {inputs['question']}\nFACTS: {outputs['context']}"
    grade = retriever_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": text},
    ])
    return {"key": "retriever_relevance", "score": int(grade["relevant"]), "comment": grade["explanation"]}

### Run Evaluation

In [11]:
EVALUATORS = [correctness,relevance,groundedness,retriever_relevance]

results = ls_client.evaluate(
    make_target(hybrid_retriever, top_k=3),
    data=DATASET_NAME,
    evaluators=EVALUATORS,
    experiment_prefix="rag-hybrid-top3",
    metadata={"retriever": "hybrid_rrf", "top_k": 3, "embedding": "bge-small-en-v1.5", "llm": "qwen/qwen3.8-27b"},
    max_concurrency=1,   # stays under Groq's free-tier rate limits
)

df = results.to_pandas()
score_cols = [c for c in df.columns if c.startswith("feedback.")]
df[score_cols].mean().round(3)

View the evaluation results for experiment: 'rag-hybrid-top3-ea74d00b' at:
https://smith.langchain.com/o/d300b7b5-5e2b-4df2-a282-77b5d2640c50/datasets/e813e690-76c1-48ca-b2ff-79771bf9da57/compare?selectedSessions=e1af4e95-3745-4f20-8c33-cddb75eb6cff




7it [03:33, 35.26s/it]Error running evaluator <DynamicRunEvaluator correctness> on run 01a0f33e-12ab-7270-9bfc-b52bb75b18f7: BadRequestError('Error code: 400 - {\'error\': {\'message\': \'Failed to parse tool call arguments as JSON\', \'type\': \'invalid_request_error\', \'code\': \'tool_use_failed\', \'failed_generation\': \'{"name": "CorrectnessGrade", "arguments": {"correct": true, "explanation": "The student answer states that feeling nervous and shaky before therapy is normal, which directly matches the ground truth’s assertion that it is normal to feel that way. The student also explains that opening up can cause uncomfortable feelings and that therapy often tackles difficult topics, which aligns with the ground truth’s points about the therapy process being scary and involving uncomfortable exploration. The student adds that over time and with trust, apprehension may lessen, which does not contradict the ground truth; it simply expands on the idea that therapy can be challenging

feedback.correctness            0.875
feedback.relevance              0.900
feedback.groundedness           1.000
feedback.retriever_relevance    1.000
dtype: float64

In [12]:
results.to_pandas()

,inputs.question,outputs.response,outputs.retrieved_ids,outputs.context,error,reference.answer,reference.doc_id,feedback.correctness,feedback.relevance,feedback.groundedness,feedback.retriever_relevance,execution_time,example_id,id
0,What factors does a counselor weigh when decid...,A counselor weighs several factors when decidi...,"[cc_q895_a012, cc_q895_a015, cc_q895_a013]",Question: How does a counselor decide when to ...,None,"In general, I usually let the client decide wh...",cc_q895_a013,1.0,1,1,1.0,0.828422,0088489d-6176-45c8-a926-033b8c8f2eba,01a0f33d-ea51-70c3-bbf7-f22745a92813
1,I'm losing my drive and feel aimless. What can...,To regain your motivation and discover a sense...,"[helios_158, cc_q94_a001, cc_q32_a002]",Question: I feel like I'm losing motivation to...,None,I'm really sorry to hear that you're feeling t...,helios_158,1.0,1,1,1.0,0.682888,1a347c7a-bccb-4654-bbd5-4e3c65449824,01a0f33d-ed8e-7b02-b186-847e265085e3
2,"I told my husband I’m feeling depressed, but h...",To get your husband to take your depression se...,"[cc_q12_a006, cc_q12_a011, cc_q12_a009]",Question: I tried telling my husband I was dep...,None,It can be hard for husbands to hear that their...,cc_q12_a011,1.0,1,1,1.0,0.744006,21956178-5a86-4f26-9d02-c32c5680b55d,01a0f33d-f03a-77e1-aefa-c8be78590fca
3,How can I tell if I’ve found the right therapi...,"To determine if you have the right therapist, ...","[cc_q933_a092, cc_q933_a081, cc_q933_a035]",Question: How do you know you have the right t...,None,Here's what I'm already picking up on from you...,cc_q933_a064,1.0,1,1,1.0,0.544505,2a6501ba-4970-4779-b0bc-dc290cbb0aa6,01a0f33d-f323-7e81-8ea4-f0e5f3251c98
4,What sets PANS/PANDAS symptoms apart from typi...,Children with PANS or PANDAS experience a very...,"[nimh_pandas_03, nimh_pandas_05, nimh_pandas_01]",Question: How do PANS/PANDAS symptoms differ f...,None,Many children with typical OCD or tics develop...,nimh_pandas_03,1.0,1,1,1.0,0.272387,406359ee-a24f-4868-a8fe-aa9b55632ebe,01a0f33d-f544-7fa0-b03a-aa3ef20a32fd
5,"I’m terrified of meeting unfamiliar people, ev...",Congratulations on your courage to ask this qu...,"[cc_q206_a003, cc_q206_a001, cc_q218_a001]",Question: I feel too scared to meet people I d...,None,Social Anxiety is scary. You don't feel safe i...,cc_q206_a001,1.0,1,1,1.0,0.531117,681a4cb0-a9f9-4a6f-9f2d-892bc9ad6435,01a0f33d-f655-76a2-8008-5786f6ea4eeb
6,I have a lot of personal challenges—sexual abu...,"No, you do not have too many issues to address...","[cc_q0_a080, cc_q0_a044, cc_q0_a011]",Question: I have so many issues to address. I ...,None,First you say I have so many issues then you s...,cc_q0_a023,1.0,1,1,1.0,6.721768,696a395a-721a-4710-adfc-d842dfec8dcf,01a0f33d-f869-77f1-a229-54263bfa9643
7,"I've been to a few couples‑therapy sessions, b...","Yes, it is completely normal to feel nervous a...","[cc_q146_a025, cc_q146_a005, cc_q146_a019]",Question: I've gone to a couple therapy sessio...,None,of I would love to know a little bit more abou...,cc_q146_a021,NaN,1,1,1.0,5.494858,9f4c5a30-f072-4a24-a175-ce76812bafc0,01a0f33e-12ab-7270-9bfc-b52bb75b18f7
8,I get angry in crowds and I can’t stand being ...,It sounds like you may be suffering from socia...,"[cc_q159_a001, cc_q159_a004, cc_q159_a010]",Question: When I'm in large crowds I get angry...,None,The thing I picked up immediately in what you'...,cc_q159_a017,0.0,1,1,NaN,3.548931,c2c24eee-d03f-40cb-8d34-0abd38276115,01a0f33e-2823-7443-851b-618bd50c05e8
9,I’m dating a guy I met online who told me he w...,"Trust is a huge thing in relationships, and fr...","[cc_q683_a001, cc_q360_a001, cc_q599_a001]",Question: I started dating a guy I met online....,None,Thank you for your question. Trust is a huge t...,cc_q683_a001,NaN,0,1,1.0,5.888388,cf1b3ce0-edbc-49ad-be30-7b0de5eb355b,01a0f33e-3600-7230-a726-cb5f86abb379
